# SQL Transactions and Isolation — Postgres Telemetry Lab

Goal: observe **READ COMMITTED**, **REPEATABLE READ**, and **SERIALIZABLE** on the live telemetry schema.

- Two sessions per demo (separate connections)
- Minimal, runnable examples


## Setup
Import the shared connection helper and define tiny query helpers.


In [1]:
from pathlib import Path
import sys
from datetime import datetime, timezone
import uuid
import time
import threading

setup_path_candidates = [
    Path.cwd() / "_setup",
    Path.cwd() / "Basics" / "Databases" / "_setup",
]
for p in setup_path_candidates:
    if p.exists():
        sys.path.append(str(p))
        break
else:
    raise FileNotFoundError("db_connections.py not found from current working directory")

from db_connections import get_postgres_conn

def fetch_one(cur, sql, params=None):
    cur.execute(sql, params or ())
    return cur.fetchone()

def fetch_val(cur, sql, params=None):
    row = fetch_one(cur, sql, params)
    return row[0] if row else None


## Telemetry sanity check


In [2]:
conn = get_postgres_conn()
cur = conn.cursor()
endpoints = fetch_val(cur, "SELECT COUNT(*) FROM telemetry.endpoints;")
alerts = fetch_val(cur, "SELECT COUNT(*) FROM telemetry.alerts;")
metrics = fetch_val(cur, "SELECT COUNT(*) FROM telemetry.metrics;")
conn.close()
print(f"endpoints: {endpoints:,} | alerts: {alerts:,} | metrics: {metrics:,}")


endpoints: 10,000 | alerts: 25,000 | metrics: 500,000


## READ COMMITTED — Non-repeatable read
Session A reads a row, Session B updates it, Session A reads again and sees a new value.


In [3]:
conn_a = get_postgres_conn()
conn_b = get_postgres_conn()
conn_a.set_session(isolation_level="READ COMMITTED")
conn_b.set_session(isolation_level="READ COMMITTED")

cur_a = conn_a.cursor()
cur_b = conn_b.cursor()

cur_a.execute("BEGIN")
ep_id, status_before = fetch_one(
    cur_a,
    "SELECT endpoint_id, status FROM telemetry.endpoints WHERE status IN ('active','inactive') LIMIT 1;"
)
print("A first read:", ep_id, status_before)

cur_b.execute("BEGIN")
orig_status = fetch_val(cur_b, "SELECT status FROM telemetry.endpoints WHERE endpoint_id = %s;", (ep_id,))
new_status = "inactive" if orig_status != "inactive" else "active"
cur_b.execute(
    "UPDATE telemetry.endpoints SET status = %s WHERE endpoint_id = %s;",
    (new_status, ep_id),
)
cur_b.execute("COMMIT")

status_after = fetch_val(cur_a, "SELECT status FROM telemetry.endpoints WHERE endpoint_id = %s;", (ep_id,))
print("A second read:", status_after)

cur_a.execute("COMMIT")

# cleanup
cur_b.execute("BEGIN")
cur_b.execute(
    "UPDATE telemetry.endpoints SET status = %s WHERE endpoint_id = %s;",
    (orig_status, ep_id),
)
cur_b.execute("COMMIT")

conn_a.close()
conn_b.close()


A first read: 50fdd579-de84-4899-a774-a3021d2cfe36 active
A second read: inactive


## READ COMMITTED — Phantom read
Session A counts rows, Session B inserts a new matching row, Session A re-counts and sees a larger set.


In [4]:
conn_a = get_postgres_conn()
conn_b = get_postgres_conn()
conn_a.set_session(isolation_level="READ COMMITTED")
conn_b.set_session(isolation_level="READ COMMITTED")

cur_a = conn_a.cursor()
cur_b = conn_b.cursor()

cur_a.execute("BEGIN")
endpoint_id = fetch_val(cur_a, "SELECT endpoint_id FROM telemetry.alerts LIMIT 1;")
count_before = fetch_val(
    cur_a,
    "SELECT COUNT(*) FROM telemetry.alerts WHERE endpoint_id = %s AND status = 'open';",
    (endpoint_id,),
)
print("A count before:", count_before)

cur_b.execute("BEGIN")
alert_id = str(uuid.uuid4())
cur_b.execute(
    """
    INSERT INTO telemetry.alerts
    (alert_id, endpoint_id, severity, message, category, status, created_at, resolved_at)
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s);
    """,
    (
        alert_id,
        endpoint_id,
        "low",
        "isolation demo (phantom)",
        "cpu",
        "open",
        datetime.now(timezone.utc),
        None,
    ),
)
cur_b.execute("COMMIT")

count_after = fetch_val(
    cur_a,
    "SELECT COUNT(*) FROM telemetry.alerts WHERE endpoint_id = %s AND status = 'open';",
    (endpoint_id,),
)
print("A count after:", count_after)

cur_a.execute("COMMIT")

# cleanup
cur_b.execute("BEGIN")
cur_b.execute("DELETE FROM telemetry.alerts WHERE alert_id = %s;", (alert_id,))
cur_b.execute("COMMIT")

conn_a.close()
conn_b.close()


A count before: 2
A count after: 3


## REPEATABLE READ — Non-repeatable read prevented
Session A holds a consistent snapshot. Session B updates a row, but Session A still sees the old value.


In [5]:
conn_a = get_postgres_conn()
conn_b = get_postgres_conn()
conn_a.set_session(isolation_level="REPEATABLE READ")
conn_b.set_session(isolation_level="READ COMMITTED")

cur_a = conn_a.cursor()
cur_b = conn_b.cursor()

cur_a.execute("BEGIN")
ep_id, status_before = fetch_one(
    cur_a,
    "SELECT endpoint_id, status FROM telemetry.endpoints WHERE status IN ('active','inactive') LIMIT 1;"
)

cur_b.execute("BEGIN")
orig_status = fetch_val(cur_b, "SELECT status FROM telemetry.endpoints WHERE endpoint_id = %s;", (ep_id,))
new_status = "inactive" if orig_status != "inactive" else "active"
cur_b.execute(
    "UPDATE telemetry.endpoints SET status = %s WHERE endpoint_id = %s;",
    (new_status, ep_id),
)
cur_b.execute("COMMIT")

status_after = fetch_val(cur_a, "SELECT status FROM telemetry.endpoints WHERE endpoint_id = %s;", (ep_id,))
print("A snapshot read:", status_before, "->", status_after)

cur_a.execute("COMMIT")

# cleanup
cur_b.execute("BEGIN")
cur_b.execute(
    "UPDATE telemetry.endpoints SET status = %s WHERE endpoint_id = %s;",
    (orig_status, ep_id),
)
cur_b.execute("COMMIT")

conn_a.close()
conn_b.close()


A snapshot read: active -> active


## REPEATABLE READ — Phantom read prevented
Session A keeps a stable row set while Session B inserts a matching row.


In [6]:
conn_a = get_postgres_conn()
conn_b = get_postgres_conn()
conn_a.set_session(isolation_level="REPEATABLE READ")
conn_b.set_session(isolation_level="READ COMMITTED")

cur_a = conn_a.cursor()
cur_b = conn_b.cursor()

cur_a.execute("BEGIN")
endpoint_id = fetch_val(cur_a, "SELECT endpoint_id FROM telemetry.alerts LIMIT 1;")
count_before = fetch_val(
    cur_a,
    "SELECT COUNT(*) FROM telemetry.alerts WHERE endpoint_id = %s AND status = 'open';",
    (endpoint_id,),
)

cur_b.execute("BEGIN")
alert_id = str(uuid.uuid4())
cur_b.execute(
    """
    INSERT INTO telemetry.alerts
    (alert_id, endpoint_id, severity, message, category, status, created_at, resolved_at)
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s);
    """,
    (
        alert_id,
        endpoint_id,
        "low",
        "isolation demo (repeatable read)",
        "cpu",
        "open",
        datetime.now(timezone.utc),
        None,
    ),
)
cur_b.execute("COMMIT")

count_after = fetch_val(
    cur_a,
    "SELECT COUNT(*) FROM telemetry.alerts WHERE endpoint_id = %s AND status = 'open';",
    (endpoint_id,),
)
print("A snapshot count:", count_before, "->", count_after)

cur_a.execute("COMMIT")

# cleanup
cur_b.execute("BEGIN")
cur_b.execute("DELETE FROM telemetry.alerts WHERE alert_id = %s;", (alert_id,))
cur_b.execute("COMMIT")

conn_a.close()
conn_b.close()


A snapshot count: 2 -> 2


## SERIALIZABLE — Write skew prevented
Both sessions observe the same predicate, both insert, one commit is rejected.


In [7]:
conn_a = get_postgres_conn()
conn_b = get_postgres_conn()
conn_a.set_session(isolation_level="SERIALIZABLE")
conn_b.set_session(isolation_level="SERIALIZABLE")

cur_a = conn_a.cursor()
cur_b = conn_b.cursor()

cur_a.execute("BEGIN")
cur_b.execute("BEGIN")

endpoint_id = fetch_val(cur_a, "SELECT endpoint_id FROM telemetry.alerts LIMIT 1;")
category = "serializable_demo"

count_a = fetch_val(
    cur_a,
    "SELECT COUNT(*) FROM telemetry.alerts WHERE endpoint_id = %s AND category = %s AND status = 'open';",
    (endpoint_id, category),
)
count_b = fetch_val(
    cur_b,
    "SELECT COUNT(*) FROM telemetry.alerts WHERE endpoint_id = %s AND category = %s AND status = 'open';",
    (endpoint_id, category),
)

alert_id_a = None
alert_id_b = None

if count_a == 0:
    alert_id_a = str(uuid.uuid4())
    cur_a.execute(
        """
        INSERT INTO telemetry.alerts
        (alert_id, endpoint_id, severity, message, category, status, created_at, resolved_at)
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s);
        """,
        (
            alert_id_a,
            endpoint_id,
            "medium",
            "serializable demo A",
            category,
            "open",
            datetime.now(timezone.utc),
            None,
        ),
    )

if count_b == 0:
    alert_id_b = str(uuid.uuid4())
    cur_b.execute(
        """
        INSERT INTO telemetry.alerts
        (alert_id, endpoint_id, severity, message, category, status, created_at, resolved_at)
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s);
        """,
        (
            alert_id_b,
            endpoint_id,
            "medium",
            "serializable demo B",
            category,
            "open",
            datetime.now(timezone.utc),
            None,
        ),
    )

try:
    conn_a.commit()
    print("A commit: OK")
except Exception as e:
    conn_a.rollback()
    print("A commit:", e.__class__.__name__)

try:
    conn_b.commit()
    print("B commit: OK")
except Exception as e:
    conn_b.rollback()
    print("B commit:", e.__class__.__name__)

# cleanup
cleanup_conn = get_postgres_conn()
cleanup_cur = cleanup_conn.cursor()
cleanup_cur.execute(
    "DELETE FROM telemetry.alerts WHERE alert_id IN (%s, %s);",
    (alert_id_a, alert_id_b),
)
cleanup_conn.commit()
cleanup_conn.close()

conn_a.close()
conn_b.close()


A commit: OK
B commit: SerializationFailure


## Deadlock example
Two sessions lock rows in opposite order. One transaction is aborted by Postgres.


In [8]:
conn = get_postgres_conn()
cur = conn.cursor()
cur.execute("SELECT endpoint_id FROM telemetry.endpoints LIMIT 2;")
row_ids = [r[0] for r in cur.fetchall()]
conn.close()

row_a, row_b = row_ids[0], row_ids[1]

def lock_then_lock(name, first_id, second_id):
    c = get_postgres_conn()
    c.set_session(isolation_level="READ COMMITTED")
    cur = c.cursor()
    cur.execute("SET deadlock_timeout = '1s';")
    cur.execute("BEGIN")
    cur.execute(
        "SELECT endpoint_id FROM telemetry.endpoints WHERE endpoint_id = %s FOR UPDATE;",
        (first_id,),
    )
    time.sleep(0.5)
    try:
        cur.execute(
            "SELECT endpoint_id FROM telemetry.endpoints WHERE endpoint_id = %s FOR UPDATE;",
            (second_id,),
        )
        c.commit()
        print(f"{name}: commit OK")
    except Exception as e:
        c.rollback()
        print(f"{name}: {e.__class__.__name__}")
    c.close()

t1 = threading.Thread(target=lock_then_lock, args=("A", row_a, row_b))
t2 = threading.Thread(target=lock_then_lock, args=("B", row_b, row_a))

t1.start()
t2.start()
t1.join()
t2.join()


B: DeadlockDetected
A: commit OK


## Why this matters in real systems

- **READ COMMITTED** is fast but can surprise you with changing reads.
- **REPEATABLE READ** is safer for analytics and audits.
- **SERIALIZABLE** is the closest to single-threaded truth, but can retry under contention.
- Deadlocks happen when services lock resources in different orders.
